# WCB label transfer

Can labelled sentences from 24 other central banks stand in for scarce FOMC
labels? Three experiments, one model (roberta-large, winner config), all text
lowercased (WCB is lowercase-only -- casing must not masquerade as register).
Results land under corpus "twd-lc": comparisons are valid only inside this
lowercased room, against the lc control, never against the cased 0.707.

1. lc control: Shah train, lowercased -> prices the casing cost
2. transfer:   WCB only (zero FOMC labels) -> Shah test
3. augment:    Shah train + WCB           -> Shah test

### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    # mounting is optional: without it, results land in the Colab session
    # filesystem and are lost when the runtime ends
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        print(f"Drive not mounted. Results will be written to {ROOT} "
              "and lost when the session ends.")

### Key Imports

In [ ]:
import torch

from config import RESULTS_DIR, SHAH_PLM
from data.loader_wcb_labelled import fetch_annotated
from data.loader_twd_labelled import load_splits
from models.plm_finetune import finetune
from utils.results import already_done, save_result
import polars as pl

OUT = RESULTS_DIR / "results.csv"
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

### Experiments 1-3

In [ ]:
WCB_SEEDS = (78516,)

cfg = SHAH_PLM["roberta-large"]
wcb = pl.from_pandas(fetch_annotated()).select(
    pl.col("sentence").str.to_lowercase(),
    pl.col("label_int").alias("label"),
)
print(f"wcb: {len(wcb):,} rows")


def lc(df):
    return df.with_columns(pl.col("sentence").str.to_lowercase())


ARMS = ["roberta-large-lc", "wcb-only:roberta-large", "wcb-aug:roberta-large"]

for arm in ARMS:
    for seed in WCB_SEEDS:
        if already_done(OUT, force=FORCE, model=arm, corpus="twd-lc", seed=seed):
            print(f"{arm} seed {seed}: already done, skipping")
            continue

        train, test = load_splits("benchmark", seed=seed)
        train, test = lc(train).select("sentence", "label"), lc(test)

        if arm == "wcb-only:roberta-large":
            train = wcb
        elif arm == "wcb-aug:roberta-large":
            train = pl.concat([train, wcb])
        print(f"{arm} seed {seed}: {len(train):,} training rows", flush=True)

        model, tok_, metrics = finetune(
            train,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=arm,
            corpus="twd-lc",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{arm} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

### Size-matched control

Borrowed pools are roughly 12 to 30 times larger than the targets' own label
sets, so quantity is confounded with source. This samples the borrowed pool down
to exactly the target's own training size, stratified by label, and refits.
Everything else is held fixed.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from data.loader_wcb_labelled import fetch_annotated

wcb_full = fetch_annotated(verbose=False)
wcb_full = wcb_full.assign(
    sentence=wcb_full["sentence"].str.lower(), label=wcb_full["label_int"]
)[["bank_name", "sentence", "label"]]


def matched(pool, n, seed):
    """Stratified sample of n rows, so class balance is preserved."""
    parts = []
    for lab, g in pool.groupby("label"):
        k = max(1, round(n * len(g) / len(pool)))
        parts.append(g.sample(n=min(k, len(g)), random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed)


# (tag, corpus, own train frame, test frame, borrowed pool)
TARGETS = []

tr, te = load_splits("benchmark", seed=WCB_SEEDS[0])
tr, te = lc(tr).to_pandas(), lc(te).to_pandas()
TARGETS.append(("fomc", "twd-lc", tr, te, wcb_full))

for tag, bank in [("ecb", "ecb"), ("boe", "bank_of_england")]:
    own = wcb_full[wcb_full["bank_name"] == bank]
    o_tr, o_te = train_test_split(
        own, test_size=0.2, random_state=WCB_SEEDS[0], stratify=own["label"]
    )
    TARGETS.append((tag, f"{tag}-lc", o_tr, o_te, wcb_full[wcb_full["bank_name"] != bank]))

for tag, corpus, own_tr, test, pool in TARGETS:
    model_key = "borrowed-matched:roberta-large"
    if already_done(OUT, force=FORCE, model=model_key, corpus=corpus, seed=WCB_SEEDS[0]):
        print(f"{corpus} {model_key}: already done, skipping")
        continue
    train = matched(pool, len(own_tr), WCB_SEEDS[0])[["sentence", "label"]]
    print(f"{corpus}: own {len(own_tr):,} -> borrowed matched {len(train):,}", flush=True)
    model, tok_, metrics = finetune(
        train,
        model_name=cfg["model_name"],
        lr=cfg["lr"],
        batch_size=cfg["batch_size"],
        seed=WCB_SEEDS[0],
        test_df=test[["sentence", "label"]],
        device=DEVICE,
        verbose=True,
    )
    save_result(
        OUT,
        model=model_key,
        corpus=corpus,
        seed=WCB_SEEDS[0],
        epochs=metrics["epochs"],
        weighted_f1=round(metrics["test_f1"], 4),
        macro_f1=round(metrics["test_macro_f1"], 4),
    )
    print(f"{corpus} {model_key}: macro={metrics['test_macro_f1']:.4f}")
    del model, tok_
    torch.cuda.empty_cache()
